Notebook for composite Bayesian Optimization on Tilt and Thickness<br>
Written by Dasol Yoon and Poompol Buathong<br>
Last edited on Jan 6, 2025

Different number of patches

In [1]:
import sys
print(sys.executable)
print(sys.version)
print(sys.version_info)

/opt/anaconda3/envs/bott-py39/bin/python
3.9.21 (main, Dec 11 2024, 10:21:40) 
[Clang 14.0.6 ]
sys.version_info(major=3, minor=9, micro=21, releaselevel='final', serial=0)


In [2]:
%matplotlib inline

In [3]:
import torch
import random
from gpytorch.mlls import ExactMarginalLogLikelihood
from botorch.fit import fit_gpytorch_model
from botorch.acquisition.objective import GenericMCObjective
from botorch.models.gp_regression import FixedNoiseGP, SingleTaskGP
from botorch.acquisition.monte_carlo import qExpectedImprovement
from botorch.acquisition import ExpectedImprovement
from botorch.optim import optimize_acqf

import math
import numpy as np
import matplotlib.pyplot as plt

# use a GPU if available
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")
dtype = torch.double
print(device)

/opt/anaconda3/envs/bott-py39/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps
mps


In [4]:
from ase.io import read
from abtem import *

In [5]:
tkwargs = {"dtype":torch.double}#,
           #"device": torch.device(device)}

In [6]:
tkwargs

{'dtype': torch.float64}

In [7]:
import scipy.ndimage
import glob
from datetime import datetime
import warnings
from PIL import Image

In [8]:
#change the file path to where the dataset is/will be stored

fpath = '/Users/pbuathong/bott/results'
#fpath = '/Users/dash/Documents/Cornell/BOtt/simulation/'

In [9]:
#load the crystal unitcell
atoms = read('/Users/pbuathong/Desktop/BoTT_data/SrTiO3.cif')
print("x,y,z extent of the unit cell ($\AA$): ",atoms.cell)
dx = atoms.cell[2,2] #step in thickness in angstrom
print('Thickness step: ',dx,' ($\AA$)')

x,y,z extent of the unit cell ($\AA$):  Cell([3.91270131, 3.91270131, 3.91270131])
Thickness step:  3.91270131  ($\AA$)


In [10]:
def simulate(x,tX,tY):
    #----variables----
    #x: thickness in angstrom
    #start = datetime.now()
    
    numUC = int(x/dx)#number of unit cells in z direction (variable x)
    #tag = '/vac_conv19p1_volt200kV_tilt0each_{:04d}.tiff'.format(int(numUC*dx))
    sc = atoms * (16,16,numUC) #todo: may need to do the rounding instead
    #sc.center(axis=(0,1),vacuum=10)
    
    #fp = FrozenPhonons(sc,20,{'Sr':.1,'Ti':.1,'O':.1},seed=1)
    fp = FrozenPhonons(sc,20,{'Sr':.088,'Ti':.0746,'O':.0963},seed=1)
    potential = Potential(fp,gpts=512, projection='infinite',#
                          #sampling=0.15, projection='infinite', #,sampling=0.06
                          slice_thickness= 2,storage  = 'gpu', precalculate=True,
                          device='gpu', parametrization='kirkland')
    probe = Probe(energy=200e3, semiangle_cutoff=19.1,tilt=(tX,tY),
                  device='gpu')
    probe.grid.match(potential)
    # print(probe.cutoff_scattering_angles)
    
    pixelated_detector = PixelatedDetector(max_angle=45) #temporary limit for thickness
 #29.345 33.258 
    gridscan = GridScan(start=[29.345,29.345], end=[33.258 ,33.258],sampling=.3)
    #scan = GridScan(start=[19.7,19.7], end=[27.6,27.6],sampling=.2)

    pixelated_measurement = probe.scan(gridscan, pixelated_detector, potential,pbar=False)
    pacbed = np.mean(pixelated_measurement.array,axis=(0,1)).astype(np.float32)
    
    #end = datetime.now() 
    # print('Time elapsed (hh:mm:ss.ms):  {}  || thickness: {} $\AA$  || shape: {}'.format(end-start,
    #                                                                                     numUC*dx,
                                                                                        # pacbed.shape))
    
    return pacbed

In [11]:
def load(x):
    #----variables----
    #x: [[thickness [angstrom], tilt]]
    #returns pre-simulated PACBED image.
    thickness=x[0,0].item()
    tilt1 = x[0,1].item()   #20250312: make it into int here?
    tilt2 = x[0,2].item()   

    numUC = int(thickness/dx)#number of unit cells in z direction (variable x)
    tag = '{:04d}.tiff'.format(int(numUC*dx))
    imgPath = 'TiltX_{}_TiltY_{}_Thickness_'.format(int(tilt1),int(tilt2)) +tag
    
    try: #load PACBED from the directory
        pacbed = plt.imread(imgPath)
        pacbed = pacbed.astype(np.float32) #added 20241112. bad simulation. crop zeroes.
    except: #simulate
        #print('no such file: ' +imgPath)
        pacbed = simulate(thickness,tilt1,tilt2)
        pacbedImg = Image.fromarray(pacbed)
        pacbedImg.save(fpath+imgPath)

    #pacbed_sqrt = np.sqrt(pacbed) #added 20241017
    #pacbed_norm = pacbed_sqrt/np.mean(pacbed_sqrt) #added 20241017; "standardize" the data...? Isn't it redundant with the normalization?
    pacbed_norm = (pacbed-np.min(pacbed))/(np.max(pacbed)-np.min(pacbed)) #normalize 0-1
    #pacbed_smol = scipy.ndimage.zoom(pacbed_norm.astype(np.float64),20/pacbed_norm.shape[0],order=1) #Nov12: comment out
    #pacbed_norm = (pacbed_smol-np.min(pacbed_smol))/(np.max(pacbed_smol)-np.min(pacbed_smol)) #normalize 0-1 #Nov12: comment out
    

    #unicode for angstrom
    #print('thickness: {} \u212B  || shape: {} || filename: '.format(x, pacbed_norm.shape)+imgPath[90:])
    
    return torch.Tensor(pacbed_norm)#.reshape(-1) #PB: change to tensor

In [12]:
start = datetime.now()
load(np.array([[250,5,-10]]))
end = datetime.now()
print(end-start)

AttributeError: 'NoneType' object has no attribute 'get_default_memory_pool'

In [ ]:
break

In [ ]:
# This is the correct value of the parameter
x_real = np.array([[310,5,-10]])

# Get the PACBED at the thickness of x_real
y_raw = load(x_real) #[simulate(x_val) for x_val in x_obs]

# Upper and lower bounds for our search
lower = 10
upper = 500

In [ ]:
temp = (
            draw_sobol_samples(
                bounds=torch.tensor([[lower,-10,-10], [upper,10,10]]).to(torch.double),
                n=n_initial_points,
                q=1,
            )
        ).to(**tkwargs)

In [ ]:
temp2 = temp[0,...]
temp2

In [ ]:
x_real[0,0].item()

In [ ]:
temp2[0,0].item()

In [ ]:
temp_y = load( np.array( [[410,1]]) )

#### testing

In [ ]:
y_raw[20:-20,20:-20].shape

In [ ]:
110**2

In [ ]:
150**2-110**2

In [ ]:
110**2/(150**2-110**2)

In [ ]:
plt.figure()
plt.imshow(y_raw[20:-20,20:-20])

In [ ]:
mask = np.zeros((150,150)).astype(bool)
mask[20:-20,20:-20] = True

In [ ]:
plt.imshow(mask)

In [ ]:
len(y_raw[~mask])

#### back to the code

In [ ]:
def twoVertTiles(y_raw):
    arr = torch.Tensor([]);
    ind = y_raw.shape[0]//2
    arr = torch.cat((arr, torch.mean(y_raw[:,:ind]).unsqueeze(-1) *1e1), dim=-1)
    arr = torch.cat((arr, torch.mean(y_raw[:,ind:]).unsqueeze(-1) *1e1), dim=-1)
    return arr

In [ ]:
def domainKnowledgeTile(y_raw):
    #create a mask to divide the image into central and peripheral regions
    ind = y_raw.shape[0]
    mask = np.zeros((ind,ind)).astype(bool)
    mask[70:-70,70:-70] = True #central region of the image #todo systematic way: CHT?
    
    arr = torch.Tensor([]);
    arr = torch.cat((arr, torch.mean(y_raw[mask]).unsqueeze(-1) *1e1), dim=-1)
    arr = torch.cat((arr, torch.mean(y_raw[~mask]).unsqueeze(-1) *1e1), dim=-1)
    return arr

In [ ]:
#TODO: use thinner one for test case
#PB changed to tensor
#TODO: Currently hard coded for the patch size. Need to modify later
def squareTiles(y_raw, **kwargs):
    numTiles= kwargs.get('numTiles',1) #default numTiles value
    arr = torch.Tensor([]);
    pixPerTile = y_raw.shape[0]//numTiles
    for i in range(numTiles):
        for j in range(numTiles):
            tile = torch.mean(y_raw[...,pixPerTile*i:pixPerTile*(i+1),...,pixPerTile*j:pixPerTile*(j+1)]).unsqueeze(-1)
            arr = torch.cat((arr,tile*1e1),dim=-1)
    return arr

In [ ]:
# #TODO: use thinner one for test case
# #PB changed to tensor
# #TODO: Currently hard coded for the patch size. Need to modify later
# def intoPatches(y_pred_raw, numPat,numPx):
#     arr = torch.Tensor([]);
#     for i in range(numPat):
#         for j in range(numPat):
#             patch = torch.mean(y_pred_raw[...,numPx*i:numPx*(i+1),...,numPx*j:numPx*(j+1)]).unsqueeze(-1)
#             arr = torch.cat((arr,patch*1e1),dim=-1)
#     return arr #unpack part ss

In [ ]:
def pixelSSE(y_cand_raw,y_ref_raw):
    err = y_cand_raw-y_ref_raw # #one image y_obs: image I want to measure
    return err.pow(2).sum().unsqueeze(-1) #np.power(err,2).sum().unsqueeze(-1) 

In [ ]:
def tileSSE(y_cand_tiles,y_ref_tiles): #patches of y and RSS in 10 variables
    return (y_cand_tiles-y_ref_tiles).pow(2).sum().unsqueeze(-1)

In [ ]:
def computeY(y_cand_raw,y_ref_raw, method, **kwargs):
    y_cand_tiles = method(y_cand_raw,**kwargs)
    y_ref_tiles = method(y_ref_raw,**kwargs)
    tileTerm = tileSSE(y_cand_tiles,y_ref_tiles)
    pixelTerm = pixelSSE(y_cand_raw, y_ref_raw)
    return torch.cat((y_cand_tiles,pixelTerm-tileTerm),dim=-1).unsqueeze(0).to(**tkwargs) #y_pred

##### testing

In [ ]:
tempTiles = domainKnowledgeTile(y_raw)  # compute reference values for patches

In [ ]:
tempTiles

In [ ]:
tempTerm = computeY(y_raw, y_raw,twoVertTiles)  # compute reference values

In [ ]:
tempTerm

In [ ]:
computeY(y_raw,y_raw,twoVertTiles)

In [ ]:
computeY(temp_y,y_raw,twoVertTiles)

In [ ]:
def tempFunc(y_pred,y_ref): #TODO: using pixelSSE(y_pred_raw) may enhance the speed a bit?
    return -((y_pred[...,:-1]-y_ref[...,:-1]).pow(2).sum()+y_pred[...,-1]).unsqueeze(-1)
    # return - (np.power( y_pred[:9]-y_obsC[:9], 2).sum() + y_pred[9] ) #pixelTerm - patchTerm #question: isn't it just the pixelTerm?

In [ ]:
tempFunc(computeY(temp_y,y_raw,twoVertTiles),computeY(y_raw,y_raw,twoVertTiles))

In [ ]:
tempFunc(computeY(y_raw,y_raw,twoVertTiles),computeY(temp_y,y_raw,twoVertTiles))

In [ ]:
pixelSSE(temp_y,y_raw)

In [ ]:
def computeObjectiveC(y_pred): #TODO: using pixelSSE(y_pred_raw) may enhance the speed a bit?
    return -((y_pred[...,:-1]-y_obsC[...,:-1]).pow(2).sum()+y_pred[...,-1]).unsqueeze(-1)
    # patchTerm +pixelTerm-patchTerm

In [ ]:
def classicalObj(x):
    return - pixelSSE(load(x))

### todo: plot
NOTE: It doesn't make sense to plot the `computeObjectiveC` because it's just getting the pixel SSE.

In [ ]:
#patch size 3
arr_obj3 = np.array([])
for i in range(10,1000,10): #start stop step
    arr_obj3 = np.append(arr_obj3, patchesSSE(intoPatches(load( np.array([[i,4]]) ))) )
obj_real = patchesSSE(y_patC)

In [ ]:
#patch size 3
#fig,ax = plt.subplots(figsize=(6,4))
plt.plot(range(10,1000,10), arr_obj3, 'o:')
plt.xlabel('thickness (Angstrom)')
plt.ylabel('objective: SSE')
plt.plot(x_real[0,0],obj_real,'ro')
plt.title('numPatches: {}'.format(numPatches))

In [ ]:
arr_obj3

### back to the code

In [ ]:
def add_data(new_x,y_ref_raw, x=None, y=None, obj=None):

    if x is None:
        x = torch.tensor([],**tkwargs)
    if y is None:
        y = torch.tensor([],**tkwargs)
    if obj is None:
        obj = torch.tensor([],**tkwargs)
    new_y = load(new_x) # y is a tensor
    new_obj = - pixelSSE(new_y,y_ref_raw) # this is our objective

    # x and obj are both 2D tensors that are nx1
    # y is a 3D tensor that is nx (numPatches**2 + 1) 
    x = torch.cat((x, new_x),dim=0)
    y = torch.cat((y, new_y),dim=0)
    obj = torch.cat((obj, torch.tensor(new_obj.clone().detach().unsqueeze(0))),dim=0)
    return x.to(**tkwargs), y.to(**tkwargs), obj.to(**tkwargs)

In [ ]:
def add_dataC(new_x,y_ref_raw,method, x=None, y=None, obj=None,**kwargs):

    if x is None:
        x = torch.tensor([],**tkwargs)
    if y is None:
        y = torch.tensor([],**tkwargs)
    if obj is None:
        obj = torch.tensor([],**tkwargs)
    y_cand_raw = load(new_x)
    new_y = computeY(y_cand_raw,y_ref_raw,method,**kwargs) # y is a tensor
    new_obj = -pixelSSE(y_cand_raw,y_ref_raw).unsqueeze(-1) #computeObjectiveC(new_y) # this is our objective

    # x and obj are both 2D tensors that are nx1
    # y is a 3D tensor that is nx (numPatches**2 + 1) 
    x = torch.cat((x, new_x),dim=0)
    y = torch.cat((y, new_y),dim=0)
    obj = torch.cat((obj, torch.tensor(new_obj.clone().detach())),dim=0)
    return x.to(**tkwargs), y.to(**tkwargs), obj.to(**tkwargs)

In [ ]:
from botorch.sampling.normal import IIDNormalSampler #import package
from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.utils.sampling import draw_sobol_samples

In [ ]:
# Plot the posterior for a single simulation output y
def plot_posterior_classic(ax, model, gridX1, gridX2, train_x, train_y, legend=True):
    with torch.no_grad(): # no need for gradients
        test_x = torch.stack([gridX1.flatten(),gridX2.flatten()],dim=1)
        # compute posterior
        posterior_x_domain = model.posterior(test_x.to(**tkwargs))

        inner_sample = IIDNormalSampler(sample_shape=torch.Size([2048])) #define based sample
        samples = inner_sample(posterior_x_domain) # do the sample
        
        samples_obj = -pixelSSE(samples) #DY: make it objective only
        
        samplesmedian,_ = samples_obj.median(dim=0) # compute median (you can use mean)
        samples025 = samples_obj.quantile(q=0.25,dim=0) #compute quantile
        samples0975=samples_obj.quantile(q=0.975,dim=0) #compute quantile
         
        # Plot training points as black stars
        ax.plot(train_x[:, 0].cpu().numpy(), 
                train_x[:, 1].cpu().numpy(), 
                -pixelSSE(train_y).cpu().numpy().squeeze(),'k*')

        # Plot posterior means as a surface 
        post = ax.plot_surface(gridX1, gridX2, 
                               samplesmedian.cpu().reshape(gridX1.shape), #.detach() is not changing the device
                               cmap='autumn_r', alpha=0.5)
        
        # Upper and lower bounds
        ax.plot_wireframe(gridX1,gridX2,
                          samples025.cpu().reshape(gridX1.shape),
                          alpha=0.5) #lw=0.5, rstride=2, cstride=2,
        ax.plot_wireframe(gridX1,gridX2,
                          samples0975.cpu().reshape(gridX1.shape),alpha=0.5)
        ax.set_xlabel('thickness');ax.set_ylabel('tilt')
        
    if legend:
        plt.legend(['Observed Data', 'Mean', 'Credible Interval'])

In [ ]:
# Plot the posterior for a single simulation output y
def plot_posterior_composite(ax, model, gridX1, gridX2, train_x, train_y, legend=True):
    with torch.no_grad(): # no need for gradients
        test_x = torch.stack([gridX1.flatten(),gridX2.flatten()],dim=1)
        # compute posterior
        posterior_x_domain = model.posterior(test_x.to(**tkwargs))

        inner_sample = IIDNormalSampler(sample_shape=torch.Size([2048])) #define based sample
        samples = inner_sample(posterior_x_domain) # do the sample
        
        samples_obj = g(samples) #DY: make it objective only
        
        samplesmedian,_ = samples_obj.median(dim=0) # compute median (you can use mean)
        samples025 = samples_obj.quantile(q=0.25,dim=0) #compute quantile
        samples0975=samples_obj.quantile(q=0.975,dim=0) #compute quantile
         
        # Plot training points as black stars
        ax.plot(train_x[:, 0].cpu().numpy(), 
                train_x[:, 1].cpu().numpy(), 
                g(train_y).cpu().numpy().squeeze(),'k*')

        # Plot posterior means as a surface 
        post = ax.plot_surface(gridX1, gridX2, 
                               samplesmedian.cpu().reshape(gridX1.shape), #.detach() is not changing the device
                               cmap='autumn_r', alpha=0.5)
        
        # Upper and lower bounds
        ax.plot_wireframe(gridX1,gridX2,
                          samples025.cpu().reshape(gridX1.shape),
                          alpha=0.5) #lw=0.5, rstride=2, cstride=2,
        ax.plot_wireframe(gridX1,gridX2,
                          samples0975.cpu().reshape(gridX1.shape),alpha=0.5)
        ax.set_xlabel('thickness');ax.set_ylabel('tilt')
        
    if legend:
        plt.legend(['Observed Data', 'Mean', 'Credible Interval'])

In [ ]:
def generate_initial_data(n_initial_points,y_ref_raw,method,**kwargs):    
    new_x = (
            draw_sobol_samples(
                bounds=torch.tensor([[lower,-10,-10], [upper,10,10]]).to(torch.double),
                n=n_initial_points,
                q=1,
            )
        ).to(**tkwargs)
    for i in range(n_initial_points):
        if i == 0:
            x,y,obj = add_dataC(new_x[i,...],y_ref_raw,method,**kwargs)
        else:
            x,y,obj = add_dataC(new_x[i,...],y_ref_raw,method,x,y,obj,**kwargs)
    return x,y,obj

In [ ]:
def generate_initial_dataClassic(n_initial_points,y_ref_raw):    
    new_x = (
            draw_sobol_samples(
                bounds=torch.tensor([[lower,-10,-10], [upper,10,10]]).to(torch.double),
                n=n_initial_points,
                q=1,
            )
        ).to(**tkwargs)
    for i in range(n_initial_points):
        if i == 0:
            x,y,obj = add_data(new_x[i,...],y_ref_raw)
        else:
            x,y,obj = add_data(new_x[i,...],y_ref_raw,x,y,obj)
    return x,y,obj

Test Fit model and optimizing EI

In [ ]:
from botorch.models.transforms import Standardize
from botorch.models.transforms.input import Normalize

In [ ]:
#x_domain = torch.linspace(lower, upper, 1000).unsqueeze(-1).to(torch.double)
def plot_model_EI(ax, gridX1, gridX2, model,acqf,iter_no):
    test_x = torch.stack([gridX1.flatten(),gridX2.flatten()],dim=1)

    post_at_x_domain = model.posterior(test_x)
    mean_at_x_domain = post_at_x_domain.mean
    std_at_x_domain = torch.sqrt(post_at_x_domain.variance)
    lo_at_x_domain = mean_at_x_domain-1.96*std_at_x_domain
    up_at_x_domain = mean_at_x_domain+1.96*std_at_x_domain
    
    acqf_val = acqf(x_domain.unsqueeze(-1))
    axis[5,1].set_title(plot_name[count_idx])
    axis[5,1].plot(x_domain,acqf_val.detach().numpy(),color='green',label='EI')
    axis[5,1].plot(new_point.detach().numpy(),new_point_EI.detach().numpy(),marker='*',linestyle='none', markersize=10, color='yellow',label='Candidate')
    axis[5,1].legend()
    plt.suptitle(f"Iteration {iter_no}", fontsize=14, y=0.91)  # Reduce `y` to bring closer
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust subplot grid to fit universal title
    plt.show()

In [ ]:
def plotEI(ax,gridX1,gridX2, ei, new_pt):
    test_x = torch.stack([gridX1.flatten(),gridX2.flatten()],dim=1)
    acq = ei.forward(test_x.unsqueeze(1).to(**tkwargs))
    acqValues = acq.detach().cpu().numpy().reshape(gridX1.shape)
    
    # Contour plot of EI over the 2D input space
    ct = ax.contourf(gridX1.numpy(), gridX2.numpy(), 
                     acqValues, cmap='viridis')
    plt.colorbar(ct)
    
    # Mark the new point found by the optimizer
    ax.plot(new_pt[0,0].cpu().numpy(), new_pt[0,1].cpu().numpy(), 'ro')
    
    ax.set_title('Expected Improvement (EI)')
    ax.set_xlabel('Thickness'); ax.set_ylabel('Tilt');

In [ ]:
### 20250312 todo: need to revisit objArr and bestArr

In [ ]:
#gridx1,gridx2 = torch.meshgrid(torch.linspace(lower,upper,101),torch.linspace(0,10,11))
#for ind,met in zip([3,10],[twoVertTiles, squareTiles]):
textLog = open('20250311-textLog_3var_method-3sqTiles.txt','w')

bestArr = []
objArr = []
xArr = []
timeArr = []
totimeArr = [] #total time array

met = squareTiles #twoVertTiles
        #domainKnowledgeTile
y_obsC = computeY(y_raw,y_raw,met,numTiles=3)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for seed in range(5):
        start = datetime.now()
        textLog.write(start.strftime('%Y-%m-%d %H:%M:%S'))
    
        np.random.seed(seed)
        torch.manual_seed(seed)
        random.seed(seed)

        n_initial_points = 4
        n_BO_points = 30

        # Generate initial data: several datapoints at random
        # Each one of them will be uniform between 0 and 3
        x,y,obj = generate_initial_data(n_initial_points,
                                        y_raw,met,numTiles=3) 
        g = GenericMCObjective(objective=lambda _y,**kwargs: - (torch.sum( (_y[...,:-1]-torch.tensor(y_obsC[...,:-1]).unsqueeze(0)).pow(2) , dim=-1)+_y[...,-1]) )
        best = [obj.max().detach().item()] # This will store the best value
        best_x_idx = obj.argmax()
        textLog.write('Best value (thickness,tilt1, tilt2,obj) found: {} , {}, {},{}\n'.format(x[best_x_idx,0].item(),
                                                              x[best_x_idx,1].item(),x[best_x_idx,2].item(),obj[best_x_idx]))
        textLog.write('Best value found: {}\n'.format(best[-1]))

        timeIt = []

        for i in range(n_BO_points): 
            torch.cuda.synchronize()
            midPoint1 = datetime.now()

            # Fit the model
            noise = torch.ones_like(y) * 0.0001
            model = FixedNoiseGP(x, y, noise,outcome_transform=Standardize(m=10),input_transform=Normalize(d=3)) # GP is on y, not the objective
            # model = SingleTaskGP(x, y) # GP is on y, not the objective
            fit_gpytorch_model(ExactMarginalLogLikelihood(model.likelihood, model))
            #g = GenericMCObjective(objective=lambda _y: -(_y[:,:9]-torch.tensor(y_obsC[:9])).pow(2).mean(dim=-1) +_y[:,9] )
            #... for the batch. Whatever dimension in front then the dimension that i specify.
            #From 100 initial points, chooses 50 initial best points for the gradient descent
            best_obj = g(y).max() # same as -obj.min()
            sampler = SobolQMCNormalSampler(1024)
            EI = qExpectedImprovement(model=model, best_f=best_obj, objective=g,sampler=sampler)
            new_point, new_point_EI = optimize_acqf(
                acq_function=EI,
                bounds=torch.tensor([[lower,-10,-10], [upper,10,10]]).to(**tkwargs),
                q=1,
                num_restarts=50,
                raw_samples=100,
            )
            textLog.write(f"new_point_EI val: {new_point_EI:.5}\n")
            
            torch.cuda.synchronize()
            midPoint2 = datetime.now()
            timeIt.append(midPoint2-midPoint1)
            
            best.append(obj.max().item())
            best_x_idx = obj.argmax()
            
            x,y,obj = add_dataC(new_point,y_raw,met,x,y,obj,numTiles=3)

            
            
        textLog.write('Seed {}: Best obj {} from iteration # {} at x of {} and {}\n'.format(seed,
                                                                           obj.max(),
                                                                           best_x_idx,
                                                                           x[best_x_idx][0].item(),
                                                                                 x[best_x_idx][1].item()))
        print('Seed {}: Best obj {} from iteration # {} at x of {} and {}\n'.format(seed,
                                                                           obj.max(),
                                                                           best_x_idx,
                                                                           x[best_x_idx][0].item(),
                                                                                 x[best_x_idx][1].item()))

        
        torch.cuda.synchronize()
        end = datetime.now()
        textLog.write(end.strftime('%Y-%m-%d %H:%M:%S'))
        textLog.write('\nTime elapsed (hh:mm:ss.ms):  {} \n'.format(end-start))
        print('Time elapsed (hh:mm:ss.ms):  {} \n'.format(end-start))
    
        totimeArr.append(end-start)

        bestArr.append(best)
        xArr.append(x.cpu().squeeze().numpy())
        timeArr.append(timeIt)
        objArr.append(obj.cpu().squeeze().numpy())

bestArrS = np.array(bestArr)
objArrS = np.array(objArr)
xArrS = np.array(xArr)
timeArrS = np.array(timeArr)
totimeArrS = np.array(totimeArr)

np.savez('20250311-composite_initial4_3var_results_10seeds_time_method3sqTiles.npz',
         xArr=xArrS,objArr=objArrS,bestArr = bestArrS, timeArr = timeArrS, totimeArr = totimeArrS)
torch.save(model.state_dict(),'20250311-model_3var_method3sqTiles.pth')

textLog.close()

In [ ]:
#gridx1,gridx2 = torch.meshgrid(torch.linspace(lower,upper,101),torch.linspace(0,10,11))
for ind,met in zip([3,10],[twoVertTiles, squareTiles]):
    textLog = open('20250311-textLog_3var_method-{}.txt'.format(ind),'w')
    
    bestArr = []
    objArr = []
    xArr = []
    timeArr = []
    totimeArr = [] #total time array
    
    met = twoVertTiles #twoVertTiles
            #domainKnowledgeTile
    y_obsC = computeY(y_raw,y_raw,met)
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for seed in range(5):
            start = datetime.now()
            textLog.write(start.strftime('%Y-%m-%d %H:%M:%S'))
        
            np.random.seed(seed)
            torch.manual_seed(seed)
            random.seed(seed)
    
            n_initial_points = 4
            n_BO_points = 30
    
            # Generate initial data: several datapoints at random
            # Each one of them will be uniform between 0 and 3
            x,y,obj = generate_initial_data(n_initial_points,
                                            y_raw,met) 
            g = GenericMCObjective(objective=lambda _y,**kwargs: - (torch.sum( (_y[...,:-1]-torch.tensor(y_obsC[...,:-1]).unsqueeze(0)).pow(2) , dim=-1)+_y[...,-1]) )
            best = [obj.max().detach().item()] # This will store the best value
            best_x_idx = obj.argmax()
            textLog.write('Best value (thickness,tilt1, tilt2,obj) found: {} , {}, {},{}\n'.format(x[best_x_idx,0].item(),
                                                                  x[best_x_idx,1].item(),x[best_x_idx,2].item(),obj[best_x_idx]))
            textLog.write('Best value found: {}\n'.format(best[-1]))
    
            timeIt = []

            for i in range(n_BO_points): 
                torch.cuda.synchronize()
                midPoint1 = datetime.now()
    
                # Fit the model
                noise = torch.ones_like(y) * 0.0001
                model = FixedNoiseGP(x, y, noise,outcome_transform=Standardize(m=ind),input_transform=Normalize(d=3)) # GP is on y, not the objective
                # model = SingleTaskGP(x, y) # GP is on y, not the objective
                fit_gpytorch_model(ExactMarginalLogLikelihood(model.likelihood, model))
                #g = GenericMCObjective(objective=lambda _y: -(_y[:,:9]-torch.tensor(y_obsC[:9])).pow(2).mean(dim=-1) +_y[:,9] )
                #... for the batch. Whatever dimension in front then the dimension that i specify.
                #From 100 initial points, chooses 50 initial best points for the gradient descent
                best_obj = g(y).max() # same as -obj.min()
                sampler = SobolQMCNormalSampler(1024)
                EI = qExpectedImprovement(model=model, best_f=best_obj, objective=g,sampler=sampler)
                new_point, new_point_EI = optimize_acqf(
                    acq_function=EI,
                    bounds=torch.tensor([[lower,-10,-10], [upper,10,10]]).to(**tkwargs),
                    q=1,
                    num_restarts=50,
                    raw_samples=100,
                )
                textLog.write(f"new_point_EI val: {new_point_EI:.5}\n")
                
                torch.cuda.synchronize()
                midPoint2 = datetime.now()
                timeIt.append(midPoint2-midPoint1)
                
                best.append(obj.max().item())
                best_x_idx = obj.argmax()
                
                x,y,obj = add_dataC(new_point,y_raw,met,x,y,obj)
    
                
                
            textLog.write('Seed {}: Best obj {} from iteration # {} at x of {} and {}\n'.format(seed,
                                                                               obj.max(),
                                                                               best_x_idx,
                                                                               x[best_x_idx][0].item(),
                                                                                     x[best_x_idx][1].item()))
            print('Seed {}: Best obj {} from iteration # {} at x of {} and {}\n'.format(seed,
                                                                               obj.max(),
                                                                               best_x_idx,
                                                                               x[best_x_idx][0].item(),
                                                                                     x[best_x_idx][1].item()))
    
            
            torch.cuda.synchronize()
            end = datetime.now()
            textLog.write(end.strftime('%Y-%m-%d %H:%M:%S'))
            textLog.write('\nTime elapsed (hh:mm:ss.ms):  {} \n'.format(end-start))
            print('Time elapsed (hh:mm:ss.ms):  {} \n'.format(end-start))
        
            totimeArr.append(end-start)
    
            bestArr.append(best)
            xArr.append(x.cpu().squeeze().numpy())
            timeArr.append(timeIt)
            objArr.append(obj.cpu().squeeze().numpy())
    
    bestArrS = np.array(bestArr)
    objArrS = np.array(objArr)
    xArrS = np.array(xArr)
    timeArrS = np.array(timeArr)
    totimeArrS = np.array(totimeArr)
    
    np.savez('20250311-composite_initial4_3var_results_10seeds_time_method{}.npz'.format(ind),
             xArr=xArrS,objArr=objArrS,bestArr = bestArrS, timeArr = timeArrS, totimeArr = totimeArrS)
    torch.save(model.state_dict(),'20250311-model_3var_method{}.pth'.format(ind))
    
    textLog.close()

### test run

In [ ]:
#gridx1,gridx2 = torch.meshgrid(torch.linspace(lower,upper,101),torch.linspace(0,10,11))

bestArr = []
objArr = []
xArr = []
timeArr = []
totimeArr = [] #total time array

met = twoVertTiles #twoVertTiles
        #domainKnowledgeTile
y_obsC = computeY(y_raw,y_raw,met)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for seed in range(1):
        start = datetime.now()
        print(start.strftime('%Y-%m-%d %H:%M:%S'))
    
        np.random.seed(seed)
        torch.manual_seed(seed)
        random.seed(seed)

        n_initial_points = 4
        n_BO_points = 10

        # Generate initial data: several datapoints at random
        # Each one of them will be uniform between 0 and 3
        x,y,obj = generate_initial_data(n_initial_points,
                                        y_raw,met) 
        g = GenericMCObjective(objective=lambda _y,**kwargs: - (torch.sum( (_y[...,:-1]-torch.tensor(y_obsC[...,:-1]).unsqueeze(0)).pow(2) , dim=-1)+_y[...,-1]) )
        best = [obj.max().detach().item()] # This will store the best value
        best_x_idx = obj.argmax()
        print('Best value (thickness,tilt,obj) found: {} , {}, {}\n'.format(x[best_x_idx,0].item(),
                                                              x[best_x_idx,1].item(),obj[best_x_idx]))
        print('Best value found: {}\n'.format(best[-1]))

        timeIt = []
#new_x,y_ref_raw,method
        for i in range(n_BO_points): 
            torch.cuda.synchronize()
            midPoint1 = datetime.now()

            # Fit the model
            noise = torch.ones_like(y) * 0.0001
            model = FixedNoiseGP(x, y, noise,outcome_transform=Standardize(m=3),input_transform=Normalize(d=3)) # GP is on y, not the objective
            # model = SingleTaskGP(x, y) # GP is on y, not the objective
            fit_gpytorch_model(ExactMarginalLogLikelihood(model.likelihood, model))
            #g = GenericMCObjective(objective=lambda _y: -(_y[:,:9]-torch.tensor(y_obsC[:9])).pow(2).mean(dim=-1) +_y[:,9] )
            #... for the batch. Whatever dimension in front then the dimension that i specify.
            #From 100 initial points, chooses 50 initial best points for the gradient descent
            best_obj = g(y).max() # same as -obj.min()
            sampler = SobolQMCNormalSampler(1024)
            EI = qExpectedImprovement(model=model, best_f=best_obj, objective=g,sampler=sampler)
            new_point, new_point_EI = optimize_acqf(
                acq_function=EI,
                bounds=torch.tensor([[lower,-10,-10], [upper,10,10]]).to(**tkwargs),
                q=1,
                num_restarts=50,
                raw_samples=100,
            )
            #textLog.write(f"new_point_EI val: {new_point_EI:.5}\n")
            print(f"new_point_EI val: {new_point_EI:.5}\n")
                
            x,y,obj = add_dataC(new_point,y_raw,met,x,y,obj)

            torch.cuda.synchronize()
            midPoint2 = datetime.now()
            timeIt.append(midPoint2-midPoint1)
            
            best.append(obj.max().item())
            best_x_idx = obj.argmax()
            
        # textLog.write('Seed {}: Best obj {} from iteration # {} at x of {} and {}\n'.format(seed,
        #                                                                    obj.max(),
        #                                                                    best_x_idx,
        #                                                                    x[best_x_idx][0].item(),
        #                                                                          x[best_x_idx][1].item()))
        print('Seed {}: Best obj {} from iteration # {} at x of {}, {}, and {}\n'.format(seed,
                                                                           obj.max(),
                                                                           best_x_idx,
                                                                           x[best_x_idx][0].item(),
                                                                                 x[best_x_idx][1].item(),
                                                                                         x[best_x_idx][2].item()))

        # # Plot the posterior and the EI
        # f, (ax1,ax2) = plt.subplots(1, 2, figsize=(8, 4))
        
        # # Plot the posterior
        # ax1.axis('off')
        # ax1 = plt.subplot(121, projection='3d') #row, col, index
        # plot_posterior_composite(ax1,model,gridx1,gridx2,x,y)

        # # Plot EI
        # plotEI(ax2,gridx1,gridx2,EI,new_point)
        # plt.show()
        
        torch.cuda.synchronize()
        end = datetime.now()
        # textLog.write(end.strftime('%Y-%m-%d %H:%M:%S'))
        # textLog.write('\nTime elapsed (hh:mm:ss.ms):  {} \n'.format(end-start))
        print('Time elapsed (hh:mm:ss.ms):  {} \n'.format(end-start))
    
        totimeArr.append(end-start)

        bestArr.append(best)
        xArr.append(x.cpu().squeeze().numpy())
        timeArr.append(timeIt)
        objArr.append(obj.cpu().squeeze().numpy())

bestArrS = np.array(bestArr)
objArrS = np.array(objArr)
xArrS = np.array(xArr)
timeArrS = np.array(timeArr)
totimeArrS = np.array(totimeArr)

# np.savez('20250204-composite_initial4_tilt_results_20seeds_time_numPatch{}.npz'.format(numPatches),
#          xArr=xArrS,objArr=objArrS,bestArr = bestArrS, timeArr = timeArrS, totimeArr = totimeArrS)
# torch.save(model.state_dict(),'20250204-model_numPatch{}.pth'.format(numPatches))

# textLog.close()

### archive

In [ ]:
def plot_model_EIsimple(model,acqf,iter_no):
    fig,axis = plt.subplots(2, 1, figsize=(6,4), gridspec_kw={'hspace': 0.5, 'wspace': 0.3})
    x_domain = torch.linspace(lower, upper, 100).unsqueeze(-1).to(torch.double)

    post_at_x_domain = model.posterior(x_domain)
    mean_at_x_domain = post_at_x_domain.mean
    std_at_x_domain = torch.sqrt(post_at_x_domain.variance)
    lo_at_x_domain = mean_at_x_domain-1.96*std_at_x_domain
    up_at_x_domain = mean_at_x_domain+1.96*std_at_x_domain
    # count_idx = 0
    # for i in range(5):
    #     for j in range(2):
    #         axis[i,j].set_title(plot_name[count_idx])
    #         axis[i,j].plot(x,y[...,count_idx].detach().numpy(),marker='*', linestyle='none', markersize=10, color='black')
    #         axis[i,j].plot(x_domain,mean_at_x_domain[...,count_idx].detach().numpy(),color='blue',linewidth=2,label='Prediction')
    #         axis[i,j].fill_between(x_domain.squeeze(-1),lo_at_x_domain[...,count_idx].detach().numpy(),up_at_x_domain[...,count_idx].detach().numpy(),alpha=0.2,color='blue',label='Uncertainty')
    #         axis[i,j].legend()
    #         count_idx+=1
    inner_sample = IIDNormalSampler(sample_shape=torch.Size([2048])) #define based sample
    samples = inner_sample(post_at_x_domain) # do the sample
    samples_obj = g(samples)
    samplesmedian,_ = samples_obj.median(dim=0)
    samples025 = samples_obj.quantile(q=0.25,dim=0)
    samples0975 = samples_obj.quantile(q=0.975,dim=0)
    axis[0].set_title('Objective')
    axis[0].plot(x,g(y).squeeze(0).detach().numpy(),marker='*', linestyle='none', markersize=10, color='black')
    axis[0].plot(x_domain,samplesmedian.detach().numpy(),color='red',linewidth=2,label='Prediction')
    axis[0].fill_between(x_domain.squeeze(-1),samples025.detach().numpy(),samples0975.detach().numpy(),alpha=0.2,color='red',label='Uncertainty')
    axis[0].legend()
    #count_idx+=1
    acqf_val = acqf(x_domain.unsqueeze(-1))
    axis[1].set_title('EI')
    axis[1].plot(x_domain,acqf_val.detach().numpy(),color='green',label='EI')
    axis[1].plot(new_point.detach().numpy(),new_point_EI.detach().numpy(),marker='*',linestyle='none', markersize=10, color='yellow',label='Candidate')
    axis[1].legend()
    plt.suptitle(f"Iteration {iter_no}", fontsize=12, y=0.91)  # Reduce `y` to bring closer
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust subplot grid to fit universal title
    plt.show()

In [ ]:
x

In [ ]:
obj

In [ ]:
plt.figure()
plt.plot(bestArrS.T);

In [ ]:
plt.figure()
plt.plot(bestArrS.T);

## test functions

### classical BO

In [ ]:
textLog = open('20250311-textLog_3var_classical.txt','w')
    
bestArr = []
objArr = []
xArr = []
timeArr = []
totimeArr = [] #total time array

for sd in range(5):
    seed = sd
       
    start = datetime.now()
    textLog.write(start.strftime('%Y-%m-%d %H:%M:%S'))
    
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
    
    n_initial_points = 4
    n_BO_points = 30
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        # Generate initial data: several datapoints at random
        # Each one of them will be uniform between 0 and 3
        x,y,obj = generate_initial_dataClassic(n_initial_points,y_raw)
        #g = GenericMCObjective(objective=lambda _y,**kwargs: - (torch.sum( (_y[...,:-1]-torch.tensor(y_obsC[...,:-1]).unsqueeze(0)).pow(2) , dim=-1)+_y[...,-1]) )
        best = [obj.max().detach().item()] # This will store the best value
        best_x_idx = obj.argmax()
        #print('initial points: ',x)
        # print('Best value (thickness,tilt,obj) found: {} , {}, {}\n'.format(x[best_x_idx,0].item(),
        #                                                       x[best_x_idx,1].item(),obj[best_x_idx]))
        
        timeIt = []
        
        for i in range(n_BO_points): 
            torch.cuda.synchronize()
            midPoint1 = datetime.now()
        
            # Fit the model
            noise = torch.ones_like(obj) * 0.0001
            model =  SingleTaskGP(x, obj,outcome_transform=Standardize(m=1),
                                 input_transform=Normalize(d=3)) # GP is on the objective
            # model = SingleTaskGP(x, y) # GP is on y, not the objective
            fit_gpytorch_model(ExactMarginalLogLikelihood(model.likelihood, model))

            #g = GenericMCObjective(objective=lambda _y: -(_y[:,:9]-torch.tensor(y_obsC[:9])).pow(2).mean(dim=-1) +_y[:,9] )
            #... for the batch. Whatever dimension in front then the dimension that i specify.
            #From 100 initial points, chooses 50 initial best points for the gradient descent
            best_obj = obj.argmax() # same as -obj.min()
            EI = ExpectedImprovement(model=model, best_f=obj.max())
            new_point, new_point_EI = optimize_acqf(
                acq_function=EI,
                bounds=torch.tensor([[lower,-10,-10], [upper,10,10]]).to(**tkwargs),
                q=1,
                num_restarts=50,
                raw_samples=100,
            )
                
        
            torch.cuda.synchronize()
            midPoint2 = datetime.now()
            timeIt.append(midPoint2-midPoint1)
            
            x,y,obj = add_data(new_point,y_raw,x,y,obj)
            best.append(obj.max().item())
            best_x_idx = obj.argmax()
            
            # print('It {}: Best obj {} from iteration # {} at x of {}, {}, and {}\n'.format(i,
            #                                                                    obj.max(),
            #                                                                    best_x_idx,
            #                                                                    x[best_x_idx][0].item(),
            #                                                                          x[best_x_idx][1].item(),
            #                                                                               x[best_x_idx][2].item()))
    torch.cuda.synchronize()
    end = datetime.now()
    textLog.write('Time elapsed (hh:mm:ss.ms):  {} \n'.format(end-start))
    
    totimeArr.append(end-start)
    bestArr.append(best)
    xArr.append(x.cpu().squeeze().numpy())
    timeArr.append(timeIt)
    objArr.append(obj.cpu().squeeze().numpy())
    
bestArrS = np.array(bestArr)
objArrS = np.array(objArr)
xArrS = np.array(xArr)
timeArrS = np.array(timeArr)
totimeArrS = np.array(totimeArr)

np.savez('20250311-composite_initial4_3var_results_5seeds_time_classical.npz',
         xArr=xArrS,objArr=objArrS,bestArr = bestArrS, timeArr = timeArrS, totimeArr = totimeArrS)
torch.save(model.state_dict(),'20250311-model_3var_classical.pth')

textLog.close()

In [ ]:
def classicalBO(sd):
    seed = sd
       
    start = datetime.now()
    print(start.strftime('%Y-%m-%d %H:%M:%S'))
    
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
    
    n_initial_points = 4
    n_BO_points = 30
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        # Generate initial data: several datapoints at random
        # Each one of them will be uniform between 0 and 3
        x,y,obj = generate_initial_data(n_initial_points)
        #g = GenericMCObjective(objective=lambda _y,**kwargs: - (torch.sum( (_y[...,:-1]-torch.tensor(y_obsC[...,:-1]).unsqueeze(0)).pow(2) , dim=-1)+_y[...,-1]) )
        best = [obj.max().detach().item()] # This will store the best value
        best_x_idx = obj.argmax()
        #print('initial points: ',x)
        # print('Best value (thickness,tilt,obj) found: {} , {}, {}\n'.format(x[best_x_idx,0].item(),
        #                                                       x[best_x_idx,1].item(),obj[best_x_idx]))
        
        timeIt = []
        
        for i in range(n_BO_points): 
            torch.cuda.synchronize()
            midPoint1 = datetime.now()
        
            # Fit the model
            noise = torch.ones_like(obj) * 0.0001
            model =  SingleTaskGP(x, obj,outcome_transform=Standardize(m=1),
                                 input_transform=Normalize(d=3)) # GP is on the objective
            # model = SingleTaskGP(x, y) # GP is on y, not the objective
            fit_gpytorch_model(ExactMarginalLogLikelihood(model.likelihood, model))

            #g = GenericMCObjective(objective=lambda _y: -(_y[:,:9]-torch.tensor(y_obsC[:9])).pow(2).mean(dim=-1) +_y[:,9] )
            #... for the batch. Whatever dimension in front then the dimension that i specify.
            #From 100 initial points, chooses 50 initial best points for the gradient descent
            best_obj = obj.argmax() # same as -obj.min()
            EI = ExpectedImprovement(model=model, best_f=obj.max())
            new_point, new_point_EI = optimize_acqf(
                acq_function=EI,
                bounds=torch.tensor([[lower,-10,-10], [upper,10,10]]).to(**tkwargs),
                q=1,
                num_restarts=50,
                raw_samples=100,
            )
                
        
            torch.cuda.synchronize()
            midPoint2 = datetime.now()
            timeIt.append(midPoint2-midPoint1)
            
            x,y,obj = add_data(new_point,x,y,obj)
            best.append(obj.max().item())
            best_x_idx = obj.argmax()
            
            print('It {}: Best obj {} from iteration # {} at x of {}, {}, and {}\n'.format(i,
                                                                               obj.max(),
                                                                               best_x_idx,
                                                                               x[best_x_idx][0].item(),
                                                                                     x[best_x_idx][1].item(),
                                                                                          x[best_x_idx][2].item()))
    torch.cuda.synchronize()
    end = datetime.now()
    print('Time elapsed (hh:mm:ss.ms):  {} \n'.format(end-start))

    return x,y,obj,best

In [ ]:
classicalBO(19)

In [ ]:
classicalBO(19)

In [ ]:
classicalBO(11)

### change functions

In [ ]:
def changedModel2(sd):
    seed = sd
       
    start = datetime.now()
    print(start.strftime('%Y-%m-%d %H:%M:%S'))
    
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
    
    n_initial_points = 4
    n_BO_points = 20
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        # Generate initial data: several datapoints at random
        # Each one of them will be uniform between 0 and 3
        x,y,obj = generate_initial_data(n_initial_points)
        #g = GenericMCObjective(objective=lambda _y,**kwargs: - (torch.sum( (_y[...,:-1]-torch.tensor(y_obsC[...,:-1]).unsqueeze(0)).pow(2) , dim=-1)+_y[...,-1]) )
        best = [obj.max().detach().item()] # This will store the best value
        best_x_idx = obj.argmax()
        print('initial points: ',x)
        print('Best value (thickness,tilt,obj) found: {} , {}, {}\n'.format(x[best_x_idx,0].item(),
                                                              x[best_x_idx,1].item(),obj[best_x_idx]))
        
        timeIt = []
        
        for i in range(n_BO_points): 
            torch.cuda.synchronize()
            midPoint1 = datetime.now()
        
            # Fit the model
            noise = torch.ones_like(obj) * 0.0001
            model =  SingleTaskGP(x, obj,outcome_transform=Standardize(m=1),
                                 input_transform=Normalize(d=2)) # GP is on the objective
            # model = SingleTaskGP(x, y) # GP is on y, not the objective
            fit_gpytorch_model(ExactMarginalLogLikelihood(model.likelihood, model))

            #g = GenericMCObjective(objective=lambda _y: -(_y[:,:9]-torch.tensor(y_obsC[:9])).pow(2).mean(dim=-1) +_y[:,9] )
            #... for the batch. Whatever dimension in front then the dimension that i specify.
            #From 100 initial points, chooses 50 initial best points for the gradient descent
            best_obj = obj.max() # same as -obj.min() #note: error. should be max, not argmax
            sampler = SobolQMCNormalSampler(1024)
            EI = qExpectedImprovement(model=model, best_f=best_obj, sampler=sampler)
            new_point, new_point_EI = optimize_acqf(
                acq_function=EI,
                bounds=torch.tensor([[lower,0], [upper,10]]).to(**tkwargs),
                q=1,
                num_restarts=50,
                raw_samples=100,
            )
                
            x,y,obj = add_data(new_point,x,y,obj)
        
            torch.cuda.synchronize()
            midPoint2 = datetime.now()
            timeIt.append(midPoint2-midPoint1)
            
            best.append(obj.max().item())
            best_x_idx = obj.argmax()
            
            print('It {}: Best obj {} from iteration # {} at x of {} and {}\n'.format(i,
                                                                               obj.max(),
                                                                               best_x_idx,
                                                                               x[best_x_idx][0].item(),
                                                                                     x[best_x_idx][1].item()))
    torch.cuda.synchronize()
    end = datetime.now()
    print('Time elapsed (hh:mm:ss.ms):  {} \n'.format(end-start))

    return x,y,obj,best

In [ ]:
def changedModel(sd):
    seed = sd
       
    start = datetime.now()
    print(start.strftime('%Y-%m-%d %H:%M:%S'))
    
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
    
    n_initial_points = 4
    n_BO_points = 20
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        # Generate initial data: several datapoints at random
        # Each one of them will be uniform between 0 and 3
        x,y,obj = generate_initial_data(n_initial_points)
        #g = GenericMCObjective(objective=lambda _y,**kwargs: - (torch.sum( (_y[...,:-1]-torch.tensor(y_obsC[...,:-1]).unsqueeze(0)).pow(2) , dim=-1)+_y[...,-1]) )
        best = [obj.max().detach().item()] # This will store the best value
        best_x_idx = obj.argmax()
        print('initial points: ',x)
        print('Best value (thickness,tilt,obj) found: {} , {}, {}\n'.format(x[best_x_idx,0].item(),
                                                              x[best_x_idx,1].item(),obj[best_x_idx]))
        
        timeIt = []
        
        for i in range(n_BO_points): 
            torch.cuda.synchronize()
            midPoint1 = datetime.now()
        
            # Fit the model
            noise = torch.ones_like(obj) * 0.0001
            model =  SingleTaskGP(x, obj,outcome_transform=Standardize(m=1),
                                 input_transform=Normalize(d=2)) # GP is on the objective
            # model = SingleTaskGP(x, y) # GP is on y, not the objective
            fit_gpytorch_model(ExactMarginalLogLikelihood(model.likelihood, model))

            #g = GenericMCObjective(objective=lambda _y: -(_y[:,:9]-torch.tensor(y_obsC[:9])).pow(2).mean(dim=-1) +_y[:,9] )
            #... for the batch. Whatever dimension in front then the dimension that i specify.
            #From 100 initial points, chooses 50 initial best points for the gradient descent
            best_obj = obj.argmax() # same as -obj.min() #note: error. should be max, not argmax
            sampler = SobolQMCNormalSampler(1024)
            EI = qExpectedImprovement(model=model, best_f=best_obj, sampler=sampler)
            new_point, new_point_EI = optimize_acqf(
                acq_function=EI,
                bounds=torch.tensor([[lower,0], [upper,10]]).to(**tkwargs),
                q=1,
                num_restarts=50,
                raw_samples=100,
            )
                
            x,y,obj = add_data(new_point,x,y,obj)
        
            torch.cuda.synchronize()
            midPoint2 = datetime.now()
            timeIt.append(midPoint2-midPoint1)
            
            best.append(obj.max().item())
            best_x_idx = obj.argmax()
            
            print('It {}: Best obj {} from iteration # {} at x of {} and {}\n'.format(i,
                                                                               obj.max(),
                                                                               best_x_idx,
                                                                               x[best_x_idx][0].item(),
                                                                                     x[best_x_idx][1].item()))
    torch.cuda.synchronize()
    end = datetime.now()
    print('Time elapsed (hh:mm:ss.ms):  {} \n'.format(end-start))

    return x,y,obj,best

In [ ]:
x11m,y11m,obj11m,best11m = changedModel2(11)

In [ ]:
x11,y11,obj11,best11 = changedModel(11)

In [ ]:
best11

### just the seed

In [ ]:
def changeSeed(sd):
    seed = sd
       
    start = datetime.now()
    print(start.strftime('%Y-%m-%d %H:%M:%S'))
    
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
    
    n_initial_points = 4
    n_BO_points = 20
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        # Generate initial data: several datapoints at random
        # Each one of them will be uniform between 0 and 3
        x,y,obj = generate_initial_data(n_initial_points)
        #g = GenericMCObjective(objective=lambda _y,**kwargs: - (torch.sum( (_y[...,:-1]-torch.tensor(y_obsC[...,:-1]).unsqueeze(0)).pow(2) , dim=-1)+_y[...,-1]) )
        best = [obj.max().detach().item()] # This will store the best value
        best_x_idx = obj.argmax()
        print('initial points: ',x)
        print('Best value (thickness,tilt,obj) found: {} , {}, {}\n'.format(x[best_x_idx,0].item(),
                                                              x[best_x_idx,1].item(),obj[best_x_idx]))
        
        timeIt = []
        
        for i in range(n_BO_points): 
            torch.cuda.synchronize()
            midPoint1 = datetime.now()
        
            # Fit the model
            noise = torch.ones_like(obj) * 0.0001
            model = FixedNoiseGP(x, obj, noise,outcome_transform=Standardize(m=1),
                                 input_transform=Normalize(d=2)) # GP is on the objective
            # model = SingleTaskGP(x, y) # GP is on y, not the objective
            fit_gpytorch_model(ExactMarginalLogLikelihood(model.likelihood, model))
            #g = GenericMCObjective(objective=lambda _y: -(_y[:,:9]-torch.tensor(y_obsC[:9])).pow(2).mean(dim=-1) +_y[:,9] )
            #... for the batch. Whatever dimension in front then the dimension that i specify.
            #From 100 initial points, chooses 50 initial best points for the gradient descent
            best_obj = obj.argmax() # same as -obj.min()
            sampler = SobolQMCNormalSampler(1024)
            EI = qExpectedImprovement(model=model, best_f=best_obj, sampler=sampler)
            new_point, new_point_EI = optimize_acqf(
                acq_function=EI,
                bounds=torch.tensor([[lower,0], [upper,10]]).to(**tkwargs),
                q=1,
                num_restarts=50,
                raw_samples=100,
            )
                
            x,y,obj = add_data(new_point,x,y,obj)
        
            torch.cuda.synchronize()
            midPoint2 = datetime.now()
            timeIt.append(midPoint2-midPoint1)
            
            best.append(obj.max().item())
            best_x_idx = obj.argmax()
            
            print('It {}: Best obj {} from iteration # {} at x of {} and {}\n'.format(i,
                                                                               obj.max(),
                                                                               best_x_idx,
                                                                               x[best_x_idx][0].item(),
                                                                                     x[best_x_idx][1].item()))
    torch.cuda.synchronize()
    end = datetime.now()
    print('Time elapsed (hh:mm:ss.ms):  {} \n'.format(end-start))

    return x,y,obj,best

In [ ]:
x_real

In [ ]:
x11,y11,obj11,best11 = changeSeed(11)

In [ ]:
x1,y1,obj1,best1 = changeSeed(1)

In [ ]:
best1

# archive

In [ ]:
gridx1,gridx2 = torch.meshgrid(torch.linspace(lower,upper,101),torch.linspace(0,10,11))
for p in [6,15]:#[1,2,3,5,6,10,15]:
    textLog = open('20250221-textLog_patch{}.txt'.format(p),'w')
    numPatches= p
    numPix = y_raw.shape[0]//numPatches
    textLog.write('numPatches: {}, numPix: {}\n'.format(numPatches, numPix))

    y_patC = intoPatches(y_raw, numPatches, numPix) # compute reference values for patches
    y_obsC = computeY(y_raw, numPatches, numPix) # compute reference values
    #add_dataC(new_x,numPat,numPx, x=None, y=None, obj=None):
    #generate_initial_data(n_initial_points,numPat,numPx)

    bestArr = []
    objArr = []
    xArr = []
    timeArr = []
    totimeArr = [] #total time array
    textLog.write("\n\n======Patch: {}======\n".format(numPatches))
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for seed in range(20):
            textLog.write('\n---------Seed: {}---------\n'.format(seed))
            start = datetime.now()
            textLog.write(start.strftime('%Y-%m-%d %H:%M:%S'))
        
            np.random.seed(seed)
            torch.manual_seed(seed)
            random.seed(seed)
    
            n_initial_points = 4
            n_BO_points = 20
    
            # Generate initial data: several datapoints at random
            # Each one of them will be uniform between 0 and 3
            x,y,obj = generate_initial_data(n_initial_points,
                                            numPatches, numPix)
            g = GenericMCObjective(objective=lambda _y,**kwargs: - (torch.sum( (_y[...,:-1]-torch.tensor(y_obsC[...,:-1]).unsqueeze(0)).pow(2) , dim=-1)+_y[...,-1]) )
            best = [obj.max().detach().item()] # This will store the best value
            best_x_idx = obj.argmax()
            textLog.write('Best value (thickness,tilt,obj) found: {} , {}, {}\n'.format(x[best_x_idx,0].item(),
                                                                  x[best_x_idx,1].item(),obj[best_x_idx]))
            textLog.write('Best value found: {}\n'.format(best[-1]))
    
            timeIt = []
    
            for i in range(n_BO_points): 
                torch.cuda.synchronize()
                midPoint1 = datetime.now()
    
                # Fit the model
                noise = torch.ones_like(y) * 0.0001
                model = FixedNoiseGP(x, y, noise,outcome_transform=Standardize(m=numPatches**2+1),input_transform=Normalize(d=2)) # GP is on y, not the objective
                # model = SingleTaskGP(x, y) # GP is on y, not the objective
                fit_gpytorch_model(ExactMarginalLogLikelihood(model.likelihood, model))
                #g = GenericMCObjective(objective=lambda _y: -(_y[:,:9]-torch.tensor(y_obsC[:9])).pow(2).mean(dim=-1) +_y[:,9] )
                #... for the batch. Whatever dimension in front then the dimension that i specify.
                #From 100 initial points, chooses 50 initial best points for the gradient descent
                best_obj = g(y).max() # same as -obj.min()
                sampler = SobolQMCNormalSampler(1024)
                EI = qExpectedImprovement(model=model, best_f=best_obj, objective=g,sampler=sampler)
                new_point, new_point_EI = optimize_acqf(
                    acq_function=EI,
                    bounds=torch.tensor([[lower,0], [upper,10]]).to(**tkwargs),
                    q=1,
                    num_restarts=50,
                    raw_samples=100,
                )
                textLog.write(f"new_point_EI val: {new_point_EI:.5}\n")
                    
                x,y,obj = add_dataC(new_point,numPatches,numPix,x,y,obj)
    
                torch.cuda.synchronize()
                midPoint2 = datetime.now()
                timeIt.append(midPoint2-midPoint1)
                
                best.append(obj.max().item())
                best_x_idx = obj.argmax()
                
            textLog.write('Seed {}: Best obj {} from iteration # {} at x of {} and {}\n'.format(seed,
                                                                               obj.max(),
                                                                               best_x_idx,
                                                                               x[best_x_idx][0].item(),
                                                                                     x[best_x_idx][1].item()))
            print('Seed {}: Best obj {} from iteration # {} at x of {} and {}\n'.format(seed,
                                                                               obj.max(),
                                                                               best_x_idx,
                                                                               x[best_x_idx][0].item(),
                                                                                     x[best_x_idx][1].item()))
    
            # Plot the posterior and the EI
            f, (ax1,ax2) = plt.subplots(1, 2, figsize=(8, 4))
            
            # Plot the posterior
            ax1.axis('off')
            ax1 = plt.subplot(121, projection='3d') #row, col, index
            plot_posterior_composite(ax1,model,gridx1,gridx2,x,y)
    
            # Plot EI
            plotEI(ax2,gridx1,gridx2,EI,new_point)
            plt.show()
            
            torch.cuda.synchronize()
            end = datetime.now()
            textLog.write(end.strftime('%Y-%m-%d %H:%M:%S'))
            textLog.write('\nTime elapsed (hh:mm:ss.ms):  {} \n'.format(end-start))
            print('Time elapsed (hh:mm:ss.ms):  {} \n'.format(end-start))
        
            totimeArr.append(end-start)
    
            bestArr.append(best)
            xArr.append(x.cpu().squeeze().numpy())
            timeArr.append(timeIt)
            objArr.append(obj.cpu().squeeze().numpy())

    bestArrS = np.array(bestArr)
    objArrS = np.array(objArr)
    xArrS = np.array(xArr)
    timeArrS = np.array(timeArr)
    totimeArrS = np.array(totimeArr)
    
    np.savez('20250204-composite_initial4_tilt_results_20seeds_time_numPatch{}.npz'.format(numPatches),
             xArr=xArrS,objArr=objArrS,bestArr = bestArrS, timeArr = timeArrS, totimeArr = totimeArrS)
    torch.save(model.state_dict(),'20250204-model_numPatch{}.pth'.format(numPatches))

    textLog.close()

In [ ]:
# gridx1,gridx2 = torch.meshgrid(torch.linspace(lower,upper,101),torch.linspace(0,10,11))
# for p in [5,10,1,2,3]:#[1,2,3,5,6,10,15]:
#     textLog = open('20250204-textLog_patch{}.txt'.format(p),'w')
#     numPatches= p
#     numPix = y_raw.shape[0]//numPatches
#     textLog.write('numPatches: {}, numPix: {}\n'.format(numPatches, numPix))

#     y_patC = intoPatches(y_raw, numPatches, numPix) # compute reference values for patches
#     y_obsC = computeY(y_raw, numPatches, numPix) # compute reference values
#     #add_dataC(new_x,numPat,numPx, x=None, y=None, obj=None):
#     #generate_initial_data(n_initial_points,numPat,numPx)

#     bestArr = []
#     objArr = []
#     xArr = []
#     timeArr = []
#     totimeArr = [] #total time array
#     textLog.write("\n\n======Patch: {}======\n".format(numPatches))
    
#     with warnings.catch_warnings():
#         warnings.simplefilter("ignore")
#         for seed in range(20):
#             textLog.write('\n---------Seed: {}---------\n'.format(seed))
#             start = datetime.now()
#             textLog.write(start.strftime('%Y-%m-%d %H:%M:%S'))
        
#             np.random.seed(seed)
#             torch.manual_seed(seed)
#             random.seed(seed)
    
#             n_initial_points = 4
#             n_BO_points = 20
    
#             # Generate initial data: several datapoints at random
#             # Each one of them will be uniform between 0 and 3
#             x,y,obj = generate_initial_data(n_initial_points,
#                                             numPatches, numPix)
#             g = GenericMCObjective(objective=lambda _y,**kwargs: - (torch.sum( (_y[...,:-1]-torch.tensor(y_obsC[...,:-1]).unsqueeze(0)).pow(2) , dim=-1)+_y[...,-1]) )
#             best = [obj.max().detach().item()] # This will store the best value
#             best_x_idx = obj.argmax()
#             textLog.write('Best value (thickness,tilt,obj) found: {} , {}, {}\n'.format(x[best_x_idx,0].item(),
#                                                                   x[best_x_idx,1].item(),obj[best_x_idx]))
#             textLog.write('Best value found: {}\n'.format(best[-1]))
    
#             timeIt = []
    
#             for i in range(n_BO_points): 
#                 torch.cuda.synchronize()
#                 midPoint1 = datetime.now()
    
#                 # Fit the model
#                 noise = torch.ones_like(y) * 0.0001
#                 model = FixedNoiseGP(x, y, noise,outcome_transform=Standardize(m=numPatches**2+1),input_transform=Normalize(d=2)) # GP is on y, not the objective
#                 # model = SingleTaskGP(x, y) # GP is on y, not the objective
#                 fit_gpytorch_model(ExactMarginalLogLikelihood(model.likelihood, model))
#                 #g = GenericMCObjective(objective=lambda _y: -(_y[:,:9]-torch.tensor(y_obsC[:9])).pow(2).mean(dim=-1) +_y[:,9] )
#                 #... for the batch. Whatever dimension in front then the dimension that i specify.
#                 #From 100 initial points, chooses 50 initial best points for the gradient descent
#                 best_obj = g(y).max() # same as -obj.min()
#                 sampler = SobolQMCNormalSampler(1024)
#                 EI = qExpectedImprovement(model=model, best_f=best_obj, objective=g,sampler=sampler)
#                 new_point, new_point_EI = optimize_acqf(
#                     acq_function=EI,
#                     bounds=torch.tensor([[lower,0], [upper,10]]).to(**tkwargs),
#                     q=1,
#                     num_restarts=50,
#                     raw_samples=100,
#                 )
#                 textLog.write(f"new_point_EI val: {new_point_EI:.5}\n")
                    
#                 x,y,obj = add_dataC(new_point,numPatches,numPix,x,y,obj)
    
#                 torch.cuda.synchronize()
#                 midPoint2 = datetime.now()
#                 timeIt.append(midPoint2-midPoint1)
                
#                 best.append(obj.max().item())
#                 best_x_idx = obj.argmax()
                
#             textLog.write('Seed {}: Best obj {} from iteration # {} at x of {} and {}\n'.format(seed,
#                                                                                obj.max(),
#                                                                                best_x_idx,
#                                                                                x[best_x_idx][0].item(),
#                                                                                      x[best_x_idx][1].item()))
#             print('Seed {}: Best obj {} from iteration # {} at x of {} and {}\n'.format(seed,
#                                                                                obj.max(),
#                                                                                best_x_idx,
#                                                                                x[best_x_idx][0].item(),
#                                                                                      x[best_x_idx][1].item()))
    
#             # Plot the posterior and the EI
#             f, (ax1,ax2) = plt.subplots(1, 2, figsize=(8, 4))
            
#             # Plot the posterior
#             ax1.axis('off')
#             ax1 = plt.subplot(121, projection='3d') #row, col, index
#             plot_posterior_composite(ax1,model,gridx1,gridx2,x,y)
    
#             # Plot EI
#             plotEI(ax2,gridx1,gridx2,EI,new_point)
#             plt.show()
            
#             torch.cuda.synchronize()
#             end = datetime.now()
#             textLog.write(end.strftime('%Y-%m-%d %H:%M:%S'))
#             textLog.write('\nTime elapsed (hh:mm:ss.ms):  {} \n'.format(end-start))
#             print('Time elapsed (hh:mm:ss.ms):  {} \n'.format(end-start))
        
#             totimeArr.append(end-start)
    
#             bestArr.append(best)
#             xArr.append(x.cpu().squeeze().numpy())
#             timeArr.append(timeIt)
#             objArr.append(obj.cpu().squeeze().numpy())

#     bestArrS = np.array(bestArr)
#     objArrS = np.array(objArr)
#     xArrS = np.array(xArr)
#     timeArrS = np.array(timeArr)
#     totimeArrS = np.array(totimeArr)
    
#     np.savez('20250204-composite_initial4_tilt_results_20seeds_time_numPatch{}.npz'.format(numPatches),
#              xArr=xArrS,objArr=objArrS,bestArr = bestArrS, timeArr = timeArrS, totimeArr = totimeArrS)
#     torch.save(model.state_dict(),'20250204-model_numPatch{}.pth'.format(numPatches))

#     textLog.close()

In [ ]:
textLog.close()

In [ ]:
tensor.device
model.to(device)

In [ ]:
temp= np.load('20250204-composite_initial4_tilt_results_20seeds_numPatch3.npz',allow_pickle=True)

In [ ]:
del temp

In [ ]:
temp.files

In [ ]:
temp['totimeArrs']

In [ ]:
bestArr = []
objArr = []
xArr = []
timeArr = []
totimeArr = [] #total time array
print("Patch: ", numPatches)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    gridx1,gridx2 = torch.meshgrid(torch.linspace(lower,upper,101),torch.linspace(0,10,11))
    for seed in range(3): #20
        print('\n---------Seed: {}---------'.format(seed))
        start = datetime.now()
    
        np.random.seed(seed)
        torch.manual_seed(seed)
        random.seed(seed)

        n_initial_points = 4
        n_BO_points = 20

        # Generate initial data: several datapoints at random
        # Each one of them will be uniform between 0 and 3
        x,y,obj = generate_initial_data(n_initial_points=n_initial_points)
        g = GenericMCObjective(objective=lambda _y,**kwargs: - (torch.sum( (_y[...,:-1]-torch.tensor(y_obsC[...,:-1]).unsqueeze(0)).pow(2) , dim=-1)+_y[...,-1]) )
        best = [obj.max().detach().item()] # This will store the best value
        best_x_idx = obj.argmax()
        print('Best value (thickness,tilt,obj) found: {} , {}'.format(x[best_x_idx,0].item(),
                                                              x[best_x_idx,1].item(),obj[best_x_idx]))
        print('Best value found: {}'.format(best[-1]))

        timeIt = []

        for i in range(n_BO_points): 
            torch.cuda.synchronize()
            midPoint1 = datetime.now()

            # Fit the model
            noise = torch.ones_like(y) * 0.0001
            model = FixedNoiseGP(x, y, noise,outcome_transform=Standardize(m=numPatches**2+1),input_transform=Normalize(d=2)) # GP is on y, not the objective
            # model = SingleTaskGP(x, y) # GP is on y, not the objective
            fit_gpytorch_model(ExactMarginalLogLikelihood(model.likelihood, model))
            #g = GenericMCObjective(objective=lambda _y: -(_y[:,:9]-torch.tensor(y_obsC[:9])).pow(2).mean(dim=-1) +_y[:,9] )
            #... for the batch. Whatever dimension in front then the dimension that i specify.
            #From 100 initial points, chooses 50 initial best points for the gradient descent
            best_obj = g(y).max() # same as -obj.min()
            sampler = SobolQMCNormalSampler(1024)
            EI = qExpectedImprovement(model=model, best_f=best_obj, objective=g,sampler=sampler)
            new_point, new_point_EI = optimize_acqf(
                acq_function=EI,
                bounds=torch.tensor([[lower,0], [upper,10]]).float(),
                q=1,
                num_restarts=50,
                raw_samples=100,
            )
            print(f"new_point_EI val: {new_point_EI:.5}")
                
            x,y,obj = add_dataC(new_point,x,y,obj)

            torch.cuda.synchronize()
            midPoint2 = datetime.now()
            timeIt.append(midPoint2-midPoint1)
            
            best.append(obj.max().item())
            best_x_idx = obj.argmax()
            
        print('Seed {}: Best obj {} from iteration # {} at x of {} and {}'.format(seed,
                                                                           obj.max(),
                                                                           best_x_idx,
                                                                           x[best_x_idx][0].item(),
                                                                                 x[best_x_idx][1].item()))

        # Plot the posterior and the EI
        f, (ax1,ax2) = plt.subplots(1, 2, figsize=(8, 4))
        
        # Plot the posterior
        ax1.axis('off')
        ax1 = plt.subplot(121, projection='3d') #row, col, index
        plot_posterior_composite(ax1,model,gridx1,gridx2,x,y)

        # Plot EI
        plotEI(ax2,gridx1,gridx2,EI,new_point)
        plt.show()
        
        torch.cuda.synchronize()
        end = datetime.now()
        print('Time elapsed (hh:mm:ss.ms):  {} \n'.format(end-start))
        totimeArr.append(end-start)

        bestArr.append(best)
        xArr.append(x.squeeze().numpy())
        timeArr.append(timeIt)
        objArr.append(obj.squeeze().numpy())

In [ ]:
numPatches

In [ ]:
# bestArrS = np.array(bestArr)
# objArrS = np.array(objArr)
# xArrS = np.array(xArr)
# timeArrS = np.array(timeArr)
# totimeArrS = np.array(totimeArr)

# np.savez('20250204-composite_initial4_tilt_results_20seeds_numPatch{}.npz'.format(numPatches),
#          xArr=xArrS,objArr=objArrS,bestArrS = bestArrS, timeArrS = timeArrS, totimeArrS = totimeArrS)

## archive

In [ ]:
bestArr = np.array(bestArr)

In [ ]:
#3 total patches
plt.figure()
plt.plot(bestArr.T);

### plot

In [ ]:
bestArr = np.array(bestArr)
xArr = np.array(xArr)
np.savez('20250108-composite_initial4_3patchesTotal.npz',bestArr=bestArr,xArr = xArr)

### archive

In [ ]:
plt.figure()
plt.plot(bestArr.T);

In [ ]:
bestAvg = np.mean(bestArr,axis=0)
bestStd = np.std(bestArr,axis=0)

In [ ]:
plt.figure()
plt.plot(bestAvg)
plt.fill_between(range(len(bestAvg)),bestAvg-bestStd,bestAvg+bestStd,alpha=0.2)
plt.xlabel('Iteration')
plt.ylabel('Objective')

## TODOs
20241204 <br>
- Evaluate the performance for x(thickness, tilt): is it worth using composite when tilt is added? What is the actual minimal cost of simulation? (e.g. ~4 seconds for classical BO vs. 30 sec - 1min for composite BO for 20 iterations/simulations)
- New simulated datasets. write a function to crop accordingly

## archive - other options
Plotting all 10 components

In [ ]:
x_domain = torch.linspace(lower, upper, 1000).unsqueeze(-1).to(torch.double)
def plot_model_EI(model,acqf,iter_no):
    fig,axis = plt.subplots(6, 2, figsize=(10, 15), gridspec_kw={'hspace': 0.5, 'wspace': 0.3})
    x_domain = torch.linspace(lower, upper, 1000).unsqueeze(-1).to(torch.double)

    post_at_x_domain = model.posterior(x_domain)
    mean_at_x_domain = post_at_x_domain.mean
    std_at_x_domain = torch.sqrt(post_at_x_domain.variance)
    lo_at_x_domain = mean_at_x_domain-1.96*std_at_x_domain
    up_at_x_domain = mean_at_x_domain+1.96*std_at_x_domain
    count_idx = 0
    for i in range(5):
        for j in range(2):
            axis[i,j].set_title(plot_name[count_idx])
            axis[i,j].plot(x,y[...,count_idx].detach().numpy(),marker='*', linestyle='none', markersize=10, color='black')
            axis[i,j].plot(x_domain,mean_at_x_domain[...,count_idx].detach().numpy(),color='blue',linewidth=2,label='Prediction')
            axis[i,j].fill_between(x_domain.squeeze(-1),lo_at_x_domain[...,count_idx].detach().numpy(),up_at_x_domain[...,count_idx].detach().numpy(),alpha=0.2,color='blue',label='Uncertainty')
            axis[i,j].legend()
            count_idx+=1
    inner_sample = IIDNormalSampler(sample_shape=torch.Size([2048])) #define based sample
    samples = inner_sample(post_at_x_domain) # do the sample
    samples_obj = g(samples)
    samplesmedian,_ = samples_obj.median(dim=0)
    samples025 = samples_obj.quantile(q=0.25,dim=0)
    samples0975 = samples_obj.quantile(q=0.975,dim=0)
    axis[5,0].set_title(plot_name[count_idx])
    axis[5,0].plot(x,g(y).squeeze(0).detach().numpy(),marker='*', linestyle='none', markersize=10, color='black')
    axis[5,0].plot(x_domain,samplesmedian.detach().numpy(),color='red',linewidth=2,label='Prediction')
    axis[5,0].fill_between(x_domain.squeeze(-1),samples025.detach().numpy(),samples0975.detach().numpy(),alpha=0.2,color='red',label='Uncertainty')
    axis[5,0].legend()
    count_idx+=1
    acqf_val = acqf(x_domain.unsqueeze(-1))
    axis[5,1].set_title(plot_name[count_idx])
    axis[5,1].plot(x_domain,acqf_val.detach().numpy(),color='green',label='EI')
    axis[5,1].plot(new_point.detach().numpy(),new_point_EI.detach().numpy(),marker='*',linestyle='none', markersize=10, color='yellow',label='Candidate')
    axis[5,1].legend()
    plt.suptitle(f"Iteration {iter_no}", fontsize=14, y=0.91)  # Reduce `y` to bring closer
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust subplot grid to fit universal title
    plt.show()